# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZaraTrimizi/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [20]:
!git clone https://github.com/samana-gillani/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 133 (delta 46), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 1.85 MiB | 10.60 MiB/s, done.
Resolving deltas: 100% (46/46), done.


In [21]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship/flyrank-ml-internship


In [22]:
from google.colab import userdata
from datasets import get_dataset_config_names

HF_TOKEN = userdata.get("HF_TOKEN")

configs = get_dataset_config_names(
    "FlyRank/internship-warehouse",
    token=HF_TOKEN
)

configs

['dim_clients',
 'dim_content',
 'fact_content_daily_performance',
 'fact_content_query_90d']

In [23]:
from google.colab import userdata
from datasets import load_dataset

HF_TOKEN = userdata.get("HF_TOKEN")

daily = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=HF_TOKEN
)

daily

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_rows: 78835655
    })
})

In [24]:
import pandas as pd

df = daily["train"].select(range(100000)).to_pandas()

In [25]:
df.shape

(100000, 30)

In [26]:
df.columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events']

In [27]:
df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115.0,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358.0,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34.0,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140.0,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89.0,...,0,0,0,0,0,0,0,0,0,0


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the performance of one content page for one client on one reporting date.

For my Refresh / Content Opportunity Scoring lane, I will use daily content performance data. The model will use observed daily search and engagement signals to help rank which pages should be reviewed for refresh.

For this assignment, I will explore a subset of the available data to verify the data contract and build candidate features.

In [28]:
df[["report_date", "client_hash_id", "content_hash_id"]].head()

,report_date,client_hash_id,content_hash_id
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964


In [29]:
df[["report_date", "client_hash_id", "content_hash_id"]].duplicated().sum()

np.int64(0)

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features
The model will use these observed features:

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- scroll_events

These are measurable signals available at the decision moment.

## Label / Proxy
There is no direct label indicating whether a page should be refreshed.

Instead, I would use a Refresh Opportunity Score as a proxy that ranks pages according to their need for review.

## Context
These fields provide context rather than predictive information:

- report_date
- client_hash_id
- content_hash_id

They identify when and where the observations were collected.

## Excluded
I exclude fields such as:

- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_copilot
- ai_claude
- ai_meta
- ai_other

These fields are not required for my initial content refresh model and may introduce unnecessary complexity.

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [31]:
len(df)

100000

In [32]:
df["report_date"].min(), df["report_date"].max()

(datetime.date(2025, 1, 27), datetime.date(2025, 3, 21))

In [33]:
df[df["gsc_data_available"]].shape

(100000, 30)

## Five candidate features

For my Refresh / Content Opportunity Scoring lane, I will use the following features:

1. gsc_impressions
   - Knowable at the decision moment because search impressions are already observed.

2. gsc_clicks
   - Knowable at the decision moment because click data has already been collected.

3. gsc_avg_position
   - Knowable at the decision moment because search ranking is observed before making a refresh decision.

4. ga4_sessions
   - Knowable at the decision moment because website sessions are historical measurements.

5. scroll_events
   - Knowable at the decision moment because user engagement has already been recorded.

In [34]:
feature_df = df[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "scroll_events",
    ]
]

feature_df.isnull().sum()

feature_df.head()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,30,0,3.833333,0,0
1,5,0,71.600000,0,0
2,1,0,34.000000,0,0
3,6,0,23.333333,0,0
4,5,0,17.800000,0,0


To demonstrate the idea of data leakage, I intentionally create a label-derived column.

If this column were used during model training, it would leak information about the target and produce unrealistically optimistic performance.

I remove it immediately after the demonstration because it would not be available at prediction time.

In [35]:
feature_df = feature_df.copy()

feature_df["refresh_label"] = (
    feature_df["gsc_impressions"] < 100
).astype(int)

feature_df.head()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events,refresh_label
0,30,0,3.833333,0,0,1
1,5,0,71.600000,0,0,1
2,1,0,34.000000,0,0,1
3,6,0,23.333333,0,0,1
4,5,0,17.800000,0,0,1


In [36]:
feature_df = feature_df.drop(columns=["refresh_label"])

feature_df.head()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,30,0,3.833333,0,0
1,5,0,71.600000,0,0
2,1,0,34.000000,0,0
3,6,0,23.333333,0,0
4,5,0,17.800000,0,0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data limits

This dataset contains observed search and engagement signals, but it cannot directly tell whether refreshing a page caused future improvements.

The data also does not capture editorial quality, content accuracy, or business priorities. Therefore, the model should be used as decision support rather than as a replacement for human judgment.

In [37]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.